In [2]:
import pandas as pd
import numpy as np

# 1. 데이터 불러오기
df = pd.read_csv("../data/processed/monthly_merged.csv")

# 2. 확인
print(df.shape)
print(df.columns)
df.head()

(900, 23)
Index(['sggCd', 'gu', 'contract_month', 'avg_sale_price', 'med_sale_price',
       'avg_sale_price_per_m2', 'med_sale_price_per_m2', 'sale_count',
       'avg_jeonse_deposit', 'med_jeonse_deposit', 'avg_jeonse_deposit_per_m2',
       'med_jeonse_deposit_per_m2', 'jeonse_count', 'total_rent_count',
       'monthly_count', 'monthly_ratio', 'jeonse_rate', 'gap_rate',
       'sale_growth_1m', 'jeonse_growth_1m', 'growth_gap_1m',
       'sale_volume_growth_1m', 'rent_volume_growth_1m'],
      dtype='str')


,sggCd,gu,contract_month,avg_sale_price,med_sale_price,avg_sale_price_per_m2,med_sale_price_per_m2,sale_count,avg_jeonse_deposit,med_jeonse_deposit,...,total_rent_count,monthly_count,monthly_ratio,jeonse_rate,gap_rate,sale_growth_1m,jeonse_growth_1m,growth_gap_1m,sale_volume_growth_1m,rent_volume_growth_1m
0,11110,종로구,2023-05,97098.529412,93495.0,1315.438781,1180.755760,34,59292.695238,59000.0,...,208,103,0.495192,0.600280,0.399720,NaN,NaN,NaN,NaN,NaN
1,11110,종로구,2023-06,125363.333333,118250.0,1382.048126,1282.217351,30,65890.121951,63000.0,...,160,78,0.487500,0.620928,0.379072,0.050637,0.086775,-0.036139,-0.117647,-0.230769
2,11110,종로구,2023-07,119448.181818,127500.0,1427.666364,1283.479442,22,62944.702970,55000.0,...,174,73,0.419540,0.587078,0.412922,0.033008,-0.023307,0.056315,-0.266667,0.087500
3,11110,종로구,2023-08,133262.000000,137225.0,1582.590432,1401.408319,40,59248.738739,48000.0,...,213,102,0.478873,0.532480,0.467520,0.108516,0.005424,0.103092,0.818182,0.224138
4,11110,종로구,2023-09,102655.208333,94250.0,1321.018893,1148.612117,48,57560.235849,51000.0,...,217,111,0.511521,0.578222,0.421778,-0.165281,-0.093574,-0.071707,0.200000,0.018779


In [3]:
# 위험 점수 계산에 사용할 주요 컬럼
risk_cols = [
    "jeonse_rate",
    "gap_rate",
    "sale_growth_1m",
    "jeonse_growth_1m",
    "growth_gap_1m",
    "sale_volume_growth_1m",
    "rent_volume_growth_1m",
    "monthly_ratio"
]

# 필요한 컬럼 결측치 확인
print(df[risk_cols].isnull().sum())

# 상승률 계산이 안 되는 첫 달 등 결측치 제거
model_df = df.dropna(subset=risk_cols).copy()

print("원본 데이터:", df.shape)
print("모델링 데이터:", model_df.shape)

jeonse_rate               0
gap_rate                  0
sale_growth_1m           25
jeonse_growth_1m         25
growth_gap_1m            25
sale_volume_growth_1m    25
rent_volume_growth_1m    25
monthly_ratio             0
dtype: int64
원본 데이터: (900, 23)
모델링 데이터: (875, 23)


In [4]:
def percentile_score(series, higher_is_risk=True):
    """
    각 값을 0~1 사이 분위수 점수로 변환
    higher_is_risk=True이면 값이 클수록 위험
    higher_is_risk=False이면 값이 낮을수록 위험
    """
    score = series.rank(pct=True)
    if higher_is_risk:
        return score
    else:
        return 1 - score

In [5]:
# 값이 클수록 위험한 변수들
model_df["gap_score"] = percentile_score(model_df["gap_rate"], higher_is_risk=True)
model_df["sale_growth_score"] = percentile_score(model_df["sale_growth_1m"], higher_is_risk=True)
model_df["growth_gap_score"] = percentile_score(model_df["growth_gap_1m"], higher_is_risk=True)
model_df["monthly_ratio_score"] = percentile_score(model_df["monthly_ratio"], higher_is_risk=True)

# 전세가 상승률은 실거주자 입장에서는 전세 유지 부담으로 볼 수 있음
model_df["jeonse_growth_score"] = percentile_score(model_df["jeonse_growth_1m"], higher_is_risk=True)

# 거래량 변화율은 너무 급격히 움직이면 위험 신호로 볼 수 있어서 절댓값 사용
model_df["sale_volume_change_score"] = percentile_score(
    model_df["sale_volume_growth_1m"].abs(), 
    higher_is_risk=True
)

model_df["rent_volume_change_score"] = percentile_score(
    model_df["rent_volume_growth_1m"].abs(), 
    higher_is_risk=True
)

In [6]:
model_df["user_risk_score"] = (
    0.30 * model_df["gap_score"] +
    0.25 * model_df["sale_growth_score"] +
    0.20 * model_df["growth_gap_score"] +
    0.15 * model_df["monthly_ratio_score"] +
    0.10 * model_df["jeonse_growth_score"]
)

In [7]:
model_df["investor_risk_score"] = (
    0.35 * model_df["gap_score"] +
    0.25 * model_df["growth_gap_score"] +
    0.20 * model_df["sale_growth_score"] +
    0.10 * model_df["sale_volume_change_score"] +
    0.10 * model_df["monthly_ratio_score"]
)

In [8]:
model_df["total_risk_score"] = (
    model_df["user_risk_score"] + model_df["investor_risk_score"]
) / 2

In [9]:
def make_risk_grade(score):
    q20 = score.quantile(0.2)
    q40 = score.quantile(0.4)
    q60 = score.quantile(0.6)
    q80 = score.quantile(0.8)

    def grade(x):
        if x <= q20:
            return "낮음"
        elif x <= q40:
            return "안정"
        elif x <= q60:
            return "보통"
        elif x <= q80:
            return "주의"
        else:
            return "높음"

    return score.apply(grade)

model_df["user_risk_grade"] = make_risk_grade(model_df["user_risk_score"])
model_df["investor_risk_grade"] = make_risk_grade(model_df["investor_risk_score"])
model_df["risk_grade"] = make_risk_grade(model_df["total_risk_score"])

In [10]:
risk_cut = model_df["total_risk_score"].quantile(0.75)

model_df["risk_target"] = np.where(
    model_df["total_risk_score"] >= risk_cut,
    1,
    0
)

In [12]:
# 상위 10% 경고 라벨 생성
warning_cut = model_df["total_risk_score"].quantile(0.9)

model_df["warning_flag"] = np.where(
    model_df["total_risk_score"] >= warning_cut,
    1,
    0
)

print(model_df["warning_flag"].value_counts())

check_cols = [
    "gu",
    "contract_month",
    "jeonse_rate",
    "gap_rate",
    "sale_growth_1m",
    "jeonse_growth_1m",
    "growth_gap_1m",
    "monthly_ratio",
    "user_risk_score",
    "investor_risk_score",
    "total_risk_score",
    "user_risk_grade",
    "investor_risk_grade",
    "risk_grade",
    "warning_flag",
    "risk_target"
]

model_df[check_cols].head()

warning_flag
0    787
1     88
Name: count, dtype: int64


,gu,contract_month,jeonse_rate,gap_rate,sale_growth_1m,jeonse_growth_1m,growth_gap_1m,monthly_ratio,user_risk_score,investor_risk_score,total_risk_score,user_risk_grade,investor_risk_grade,risk_grade,warning_flag,risk_target
1,종로구,2023-06,0.620928,0.379072,0.050637,0.086775,-0.036139,0.487500,0.506171,0.383314,0.444743,보통,안정,안정,0,0
2,종로구,2023-07,0.587078,0.412922,0.033008,-0.023307,0.056315,0.419540,0.502686,0.532057,0.517371,보통,보통,보통,0,0
3,종로구,2023-08,0.532480,0.467520,0.108516,0.005424,0.103092,0.478873,0.738629,0.769829,0.754229,높음,높음,높음,1,1
4,종로구,2023-09,0.578222,0.421778,-0.165281,-0.093574,-0.071707,0.511521,0.256800,0.276971,0.266886,낮음,낮음,낮음,0,0
5,종로구,2023-10,0.661237,0.338763,-0.077389,0.055069,-0.132458,0.457447,0.221771,0.171886,0.196829,낮음,낮음,낮음,0,0


In [14]:
model_df.to_csv("../data/processed/modeling_dataset.csv", index=False, encoding="utf-8-sig")

print("저장 완료: ../data/processed/modeling_dataset.csv")
print(model_df.shape)

저장 완료: ../data/processed/modeling_dataset.csv
(875, 38)
